# Pilot Task 1 — PDP ResNet50 và ConvNeXt-V2-Base

Ngày tạo: 10/10/2026, UTC+7. Chuẩn bị local, real run trên RTX 4090. Không có notebook nghiệm thu môi trường Vast.ai riêng. Hai models dùng cùng data/resolution/batch hiệu dụng/epochs, theo configs native đã nghiệm thu.

Đọc [hướng dẫn vận hành](../../docs/project/TRAINING_NOTEBOOK_RUNBOOK.md) trước khi chạy. Mọi artifacts dưới `data/training/`; mặc định `dry_run=true` dùng CPU/random ResNet18/6 labels, không phải kết quả ResNet50–ConvNeXt.

Chạy ngầm qua tmux bằng `python -m tools.execute_training_notebook --notebook notebooks/training/01_task1_pilot.ipynb --settings configs/training/notebooks.json`. Executor dùng đúng Python đang gọi, không giới hạn thời gian cell và lưu outputs sau mỗi cell.


## 1. Cấu hình và thư mục campaign
Dùng cùng settings file/campaign_id cho cả ba notebooks. Khi rerun giữ nguyên campaign, batch, epochs, data và resolution để resume. Đổi thí nghiệm cần campaign_id mới. Cấu hình quan trọng: dry_run, roots dưới data, resolution, physical/effective batch, pilot_epochs và checkpoint_every_minutes.


In [1]:
import os
os.environ["AUTOCHECKOUT_TRAINING_SETTINGS"] = (
    "/workspace/AutoCheckout-CL/data/training/settings_4090.json"
)

In [2]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout.io import load_json
from autocheckout.training_workflow import prepare, run_phase, resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'configs/training/notebooks.json'))
SETTINGS = load_json(SETTINGS_PATH)
PHASE = 'pilot'
print('Settings:', SETTINGS_PATH)
print(json.dumps(SETTINGS, indent=2))


Settings: /workspace/AutoCheckout-CL/data/training/settings_4090.json
{
  "campaign_id": "vast4090-seed0-640-v1",
  "artifact_root": "data/training",
  "dry_run": false,
  "experiments": [
    "EXP-B1",
    "EXP-B2"
  ],
  "raw_root": "data/archive",
  "release_root": "data/processed/rpc_100-4x25_seed0_native_v1",
  "pretrained_root": "data/pretrained/hf-cache",
  "resolution": 640,
  "batch_size": 1,
  "effective_batch": 2,
  "num_workers": 4,
  "pilot_epochs": 2,
  "transition_epochs": 1,
  "transition_samples": 16,
  "baseline_epochs": null,
  "checkpoint_every_minutes": 2,
  "count_nms_grid": "0.45",
  "test_interruption": false
}


In [3]:
PLAN = prepare(SETTINGS, PHASE)
print(json.dumps({k: PLAN[k] for k in ['phase_dir', 'dry_run', 'epochs', 'resolution', 'batch_size', 'effective_batch', 'accumulation', 'steps_per_epoch', 'actual_backbone']}, indent=2))
print('Estimated Task 1 optimizer steps:', PLAN['epochs'] * PLAN['steps_per_epoch'])
print('FP32, TF32 off; frozen backbone/encoder/decoder; train classifier + prompts.')


{
  "phase_dir": "/workspace/AutoCheckout-CL/data/training/vast4090-seed0-640-v1/pilot",
  "dry_run": false,
  "epochs": 2,
  "resolution": 640,
  "batch_size": 1,
  "effective_batch": 2,
  "accumulation": 2,
  "steps_per_epoch": 10876,
  "actual_backbone": "per-experiment"
}
Estimated Task 1 optimizer steps: 21752
FP32, TF32 off; frozen backbone/encoder/decoder; train classifier + prompts.


## 2. Train, interruption/resume và evaluation
Runner ghi stdout/stderr từng invocation vào attempts/<id>/; task logs gồm train.log, step_metrics.jsonl, progress.json, checkpoint_status.json và Lightning CSV. Dry run cố ý dừng ở step 1 (exit 75), rồi tiến trình mới resume. Real run lưu checkpoint mỗi 2 phút mặc định và cuối epoch; chạy lại cùng campaign để tiếp tục.

Cuối Task 1, xuất val/test predictions, raw mAP và policy chọn trên val rồi áp cho test. Pilot mặc định 2 epochs là budget khởi đầu, cần đọc loss/val và prototype coverage trước khi chọn baseline_epochs; không tuyên bố convergence từ budget này.


In [4]:
SUMMARY = run_phase(PLAN)
for case in SUMMARY['cases']:
    print(case['experiment'], json.dumps([{'task': t['task_id'], 'optimizer_steps': t['optimizer_steps'], 'seconds': t['seconds'], 'peak_gpu_gib': t['peak_gpu_gib'], 'missing_prototypes': t['prototype_diagnostics'], 'resume': t['resume_verification']} for t in case['tasks']], indent=2))
print('Status:', SUMMARY['status'], 'dry_run:', SUMMARY['dry_run'])
print('Saved summary:', Path(PLAN['phase_dir']) / 'summary.json')


2026-10-10T13:46:28+07:00 START bash /workspace/AutoCheckout-CL/scripts/run_exp.sh /workspace/AutoCheckout-CL/data/training/vast4090-seed0-640-v1/pilot/EXP-B1/resolved_config.sh -> /workspace/AutoCheckout-CL/data/training/vast4090-seed0-640-v1/pilot/attempts/258fb876-4d27-4959-88f7-6c699cfaefd3/EXP-B1_task1_0_runner.log


KeyboardInterrupt: 

## 3. Review và bước kế tiếp
Đọc loss/validation trong Lightning CSV và step_metrics.jsonl; xem metrics_cl_val.json và calibration_count.json trước khi chọn budget. Không chọn budget bằng test/oracle. Kiểm tra missing_prototypes: những lớp không có prototype được ghi rõ, không suy ra PPG phủ đủ SKU.

Sau pilot hoàn tất, chạy 02_task_transition.ipynb cùng settings. Artifacts và notebook outputs được giữ trong data để upload/download cùng campaign. Nếu chạy trực tiếp trong IDE, lưu bản notebook có outputs vào data; khi cần tránh mất SSH dùng executor trong tmux theo runbook.
